# The Model Was Running Fine. The Problem Was What Its Defaults Meant.

**LinkedIn article draft - AdaBoost regression**

## The hard part is not finding a spike

The hard part is deciding which spikes deserve human attention. A model can run without errors, produce an acceptable aggregate metric, and still create the wrong alerts. I explored that distinction with scikit-learn's `AdaBoostRegressor` on synthetic hourly website traffic. This is a controlled stress test, not production evidence or a claim about a real incident.

The task was to estimate expected traffic from hour, day of week, a marketing-event flag and weekend status. Most marketing events create a modest lift. A rare but legitimate combination - a campaign at 8pm on its event weekend - creates a much larger lift. The inputs describe that combination, but it is uncommon. I also placed a similarly sized, one-hour simulated surge in the held-out period with no explanatory input. A residual can trigger review; it cannot establish an attack.

> **EDITORIAL MARKER - upload `ai_prompt_vs_model_judgement.png` here, then remove this marker before publishing.**
>
> Caption: *Running a model is not the same as understanding its behaviour.*
>
> Alt text: A workflow from an AI-generated model and acceptable aggregate MAE to scenario-level error inspection, reweighting-policy review, and an alerting decision.

## Why this is an AdaBoostRegressor question

`AdaBoostRegressor` fits shallow regressors sequentially, and the result of one round changes which examples matter in the next. This is not merely a collection of trees. It is a reweighting process.

After each weak regressor, scikit-learn normalises absolute residuals to `e` between zero and one. With the default, `linear`, the transformed error is `L(e) = e`; with `square`, it is `L(e) = e^2`. It calculates weighted round error `E`, sets `beta = E / (1 - E)`, then updates a sample with `weight *= beta ** ((1 - L(e)) * learning_rate)`. Because beta is below one, samples with larger transformed loss are downweighted less in the following round.

The difference is concrete. At `e = 0.6`, linear gives 0.60 while square gives 0.36. At `e = 0.9`, they give 0.90 and 0.81. Square downplays middle-sized errors relative to near-maximum errors; it changes the model's attention trajectory rather than simply applying a stronger final penalty.

> **EDITORIAL MARKER - upload `adaboost_loss_reweighting.png` here, then remove this marker before publishing.**
>
> Caption: *The loss setting changes AdaBoostRegressor's next-round sample weights; it is not the final prediction metric.*
>
> Alt text: Linear and square loss transformations and their resulting illustrative sample-weight multipliers over normalised prediction error.

## The default looked fine - until I inspected the slice that mattered

I fixed the base tree at depth three, the seed, features, split, learning rate and 20 boosting rounds; only `loss` varied. There were 4,320 hourly rows and a chronological 60%/20%/20% train/validation/test split.

On the untouched test period, default linear loss had ordinary-traffic MAE of 59.52. That would look acceptable in a quick model report. But rare known campaign MAE was 700.65: expected business activity would have looked anomalous. With square loss, rare-campaign MAE fell to 41.31, while ordinary-traffic MAE was 60.81. The unexplained one-hour surge remained a large residual in both models: 1,814.33 under linear and 1,811.42 under square.

The calculation above is the mechanism, not a magic anomaly detector. The loss setting changes which training errors later trees are built to address - and that changes the operational alert outcome.

> **EDITORIAL MARKER - upload `adaboost_known_vs_unexplained.png` here, then remove this marker before publishing.**
>
> Caption: *The operational goal: learn the known campaign lift without normalising the unexplained spike.*
>
> Alt text: Side-by-side traffic charts. Square loss predicts the known campaign spike more closely than linear loss; both leave the unexplained spike as a large residual.

## The AI-agent lesson

An AI agent can instantiate the default estimator, run it and report an acceptable aggregate number. That is useful, but it is model operation, not model judgement. The meaningful contribution of a seasoned data scientist is the habit of opening the black box: inspect defaults, trace the training mechanism, define the costly error, and test the relevant operational slice. The question was not 'Did AdaBoostRegressor run?' It was 'Which legitimate events will this configuration ask someone to investigate?'

For a separate explanation of AdaBoost fundamentals, visit the [AlgoStalk YouTube channel](https://www.youtube.com/@AlgoStalk).

### References

- [scikit-learn AdaBoostRegressor documentation and source](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostRegressor.html)
- [Drucker, Improving Regressors using Boosting Techniques (1997)](https://dl.acm.org/doi/10.5555/645526.657132)

## Reproducible experiment

Run the next cell from either the repository root or this notebook's directory. `run_experiment.py` generates the synthetic data and reports MAE/RMSE for validation and held-out test slices.

In [ ]:
from pathlib import Path
import runpy

script = next(path for path in (
    Path.cwd() / 'run_experiment.py',
    Path.cwd() / 'course' / 'Trees' / 'AdaBoost' / 'run_experiment.py',
) if path.exists())
runpy.run_path(str(script))